# 04 · Character n-gram model on the raw URL

**Idea:** instead of choosing features ourselves, let the model see every short chunk of characters in the URL.

A **character n-gram** is a run of *n* consecutive characters. `paypa1.top` contains the 3-grams `pay`, `ayp`, `ypa`, `pa1`, `a1.`, `1.t`, `.to`, `top`. Each URL becomes a long, mostly-zero vector with a score for every n-gram it contains:
- **TF-IDF** weights n-grams that are common in this URL but rare overall.
- **Logistic Regression** learns one weight per n-gram.

This is the classical forerunner of the character-level CNN in Milestone 4.

Published URL-only references (Frontiers in Computer Science, 2026; different split, so indicative only): Logistic Regression 99.62%, CharCNN 99.81%.

In [1]:
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from xgboost import XGBClassifier

from phishguard.data import load_clean, TARGET
from phishguard.features import extract_features
from phishguard.ensemble import LogitAverage
from phishguard.experiment import run, summary_table, load_predictions, load_record
from phishguard.modeling import param_grid
from phishguard.stats import mcnemar

df = load_clean()
y = df[TARGET]
urls = df["URL"]

## 1. Tune and test over 5 splits

Two settings matter:
- `max_n`: the longest n-gram used (from 1 up to `max_n`)
- `C`: regularisation (smaller means simpler)

The `C` grid goes up to 30, so the best value isn't stuck at the edge of the search.

In [2]:
def make_ngram(max_n, C):
    return make_pipeline(
        TfidfVectorizer(analyzer="char", ngram_range=(1, max_n), min_df=2,
                        sublinear_tf=True, dtype=np.float32),
        LogisticRegression(C=C, solver="liblinear", max_iter=3000),
    )

ngram = run("Char n-gram LogReg (raw URL)", make_ngram, param_grid(max_n=[3, 4], C=[10.0, 30.0, 100.0]),
            urls, y, df, features="raw URL", group="m2")
ngram.per_seed[["seed", "params", "threshold", "accuracy", "fp", "fn"]]

,seed,params,threshold,accuracy,fp,fn
0,42,"{'max_n': 4, 'C': 30.0}",0.30,0.997467,7,82
1,0,"{'max_n': 3, 'C': 10.0}",0.25,0.996670,14,103
2,1,"{'max_n': 3, 'C': 30.0}",0.36,0.996195,12,121
3,2,"{'max_n': 4, 'C': 30.0}",0.27,0.997085,5,97
4,3,"{'max_n': 4, 'C': 30.0}",0.20,0.996897,10,99


## 2. What did it learn? (seed-42 model)

One weight per n-gram: large positive weights push towards phishing, large negative ones towards legitimate.

In [3]:
vectorizer, logreg = ngram.reference.model.named_steps.values()
print(f"{len(vectorizer.vocabulary_):,} distinct n-grams")
weights = pd.Series(logreg.coef_[0], index=vectorizer.get_feature_names_out())
pd.DataFrame({
    "→ phishing": weights.nlargest(20).index, "weight": weights.nlargest(20).round(1).values,
    "→ legitimate": weights.nsmallest(20).index, "weight ": weights.nsmallest(20).round(1).values,
})

306,717 distinct n-grams


,→ phishing,weight,→ legitimate,weight
0,p://,30.7,ww,-29.3
1,p:,30.7,w,-26.1
2,p:/,30.7,www.,-19.2
3,tp:,30.7,ww.,-18.6
4,tp:/,30.7,/www,-17.8
5,ttp:,30.7,//ww,-17.8
6,m/,16.0,www,-17.7
7,com/,15.0,/ww,-17.5
8,om/,15.0,w.,-17.0
9,/a,10.2,://w,-15.8


**Look at the top of both lists.**
- The strongest phishing signal is `p://`, which is `http://` *without* the `s`.
- The strongest legitimate signals are `www` and `s://`.

The model has learned mostly *how the URL starts*, not what the domain says. Every legitimate URL in PhiUSIIL starts with `https://www.`, so this works here, but a real phishing site can use `https://www.` too. That's the main question of Milestone 3. Real signals such as `.top` appear further down.

## 3. Is the n-gram model really better than XGBoost on URL features?

Both were tested on the same seed-42 test URLs. **McNemar's test** looks only at URLs where the two models disagree. If the models were equally good, those disagreements would split roughly 50/50.

In [4]:
xgb_pred = load_predictions("m2", "XGBoost (URL features)")
ngram_pred = load_predictions("m2", "Char n-gram LogReg (raw URL)")
assert (xgb_pred["test_idx"] == ngram_pred["test_idx"]).all()
truth = y.loc[ngram_pred["test_idx"]].to_numpy()
correct_ngram = (ngram_pred["test_prob"] >= ngram_pred["threshold"]) == truth
correct_xgb = (xgb_pred["test_prob"] >= xgb_pred["threshold"]) == truth
mcnemar(correct_ngram, correct_xgb)

{'only_a_right': 38, 'only_b_right': 16, 'p_value': 0.003838265880326096}

## 4. Combining it with the URL-feature model

The two models read different things: XGBoost sees counts and structure, the n-gram model sees the characters. `LogitAverage` averages their **log-odds**, which is fairer than averaging probabilities when the models are calibrated differently. It's a first preview of the hybrid model in Milestone 5.

Each part uses the settings chosen on the seed-42 split in notebooks 03 and 04.

In [5]:
xgb_settings = load_record("m2", "XGBoost (URL features)")["reference"]["params"]
ngram_settings = ngram.per_seed.set_index("seed").loc[42, "params"]
print("XGBoost:", xgb_settings, "| n-gram:", ngram_settings)

X_both = extract_features(urls).assign(URL=urls)
feature_cols = [c for c in X_both.columns if c != "URL"]
make_average = lambda: LogitAverage({
    "ngram": (lambda: make_ngram(**ngram_settings), "URL"),
    "xgb": (lambda: XGBClassifier(n_estimators=500, learning_rate=0.1, subsample=0.8, colsample_bytree=0.8,
                                  n_jobs=-1, random_state=42, **xgb_settings), feature_cols),
})
average = run("Average: n-gram + XGBoost (URL only)", make_average, [{}],
              X_both, y, df, features="raw URL + URL features", group="m2")
summary_table([ngram, average])

XGBoost: {'max_depth': 6, 'min_child_weight': 5} | n-gram: {'max_n': 4, 'C': 30.0}


,accuracy (%),f1 (%),roc_auc,pr_auc,mcc,tpr_at_fpr_1pct (%),"accuracy, seed 42 [95% CI]"
model,,,,,,,
Char n-gram LogReg (raw URL),99.69 ± 0.05,99.63 ± 0.06,0.9986 ± 0.0003,0.9988 ± 0.0002,0.9936 ± 0.0010,99.51 ± 0.08,"99.75% [99.69%, 99.80%]"
Average: n-gram + XGBoost (URL only),99.70 ± 0.03,99.64 ± 0.04,0.9986 ± 0.0002,0.9988 ± 0.0002,0.9938 ± 0.0006,99.51 ± 0.05,"99.75% [99.68%, 99.80%]"


## Result

URL-only models land at about 99.7%, comparable with the best published URL-only results on PhiUSIIL (see notebook 05 for the full comparison).

Check the standard deviations and the McNemar p-value before calling any model "better": differences of a few hundredths of a percent are within split-to-split noise. Nearly all remaining errors are phishing sites shaped like a plain `https://www.<domain>` homepage (notebook 03).